# Incident Priority Classification – Solution

**Short name (GitHub):** `CrimRisk`  
**Lab source:** MushEdib / EduRisk pipeline adapted to a criminal-incident RMS codebook.  
**Data:** `data/incidents.csv` (6,810 × 22 letter codes, synthetic). Target `priority`: **p** = priority, **r** = routine.  
**Companion files:** `CrimRisk_Solution.ipynb`, `CrimRisk_Reusable_Template.ipynb`, `CrimRisk.py`, `CrimRisk_Cheatsheet.docx`, `CrimRisk_Project_Memo.docx`, `CrimRisk_Strategy_Guide.docx`, `CrimRisk_Implementation_Checklist.docx`, `CrimRisk_1Page_Summary_Report.docx`, `crimrisk_flowchart.png`.

This notebook is the worked key. Use `CrimRisk_Practice_Skeleton.ipynb` to practice first.

**This is not bail, sentencing, charging, or person-profiling.** The unit is an *incident row*, not a person. Protected-class fields are not in the table on purpose.

You will:

1. Clean `?` in `clearance` → `u`, drop exact duplicates.
2. Plot priority balance, weapon × priority, district × priority, and a 12-feature factorize heatmap.
3. Drop zero-variance `file-flag`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.
6. Walk the implementation checklist before anyone would touch a live RMS.



## Inline cheat-sheet (keep this cell visible)

See also **`CrimRisk_Cheatsheet.docx`** and **`CrimRisk_Implementation_Checklist.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/incidents.csv")` |
| Missing | literal `"?"` in `clearance` (~1,848 rows) |
| Recode | `df["clearance"] = df["clearance"].replace("?", "u")` |
| Duplicates | expect **11** exact clones → `drop_duplicates()` |
| Zero-variance | `file-flag` is always `i` — drop |
| Factorize heatmap | drop `priority` before `.head(12)` — 12×12 not 13×13 |
| LE map | alphabetical → **`p=0`, `r=1`** (priority is class 0) |
| Split | `train_test_split(..., test_size=0.2, random_state=42)` |
| Shapes | `X_train (5439, 20)`, `X_test (1360, 20)` |
| RF | `RandomForestClassifier(random_state=42)` |
| Costly cell | actual **p**, predicted **r** (missed priority case) |
| Weapon rule | majority priority per weapon code ≈ 0.73 |
| Scale? | No. Trees split on thresholds. |

**Do not** add race, ethnicity, or neighborhood proxies to this teaching table and call it an improvement.



## Flowchart of the desired outcome

![CrimRisk flow](crimrisk_flowchart.png)

Clean first. Look at **weapon** before you fit anything. Drop `file-flag`. Freeze seed 42. Score the costly cell (missed priority), not only accuracy. Then read the implementation checklist before you talk about production.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

try:
    import CrimRisk as cr
except ImportError:
    cr = None
print("ready")


## 1. Data preparation

Load `data/incidents.csv`. Show `head()`, `.info()`, count `"?"` in `clearance`, replace with `"u"`, drop duplicates. Print cleaned shape and `priority` counts.

Expected: 6,810 raw rows × 22 columns, ~1,848 question marks, 11 duplicates → **6,799** cleaned rows. Priority split about 3,393 p / 3,406 r.


In [ ]:
df = pd.read_csv("data/incidents.csv")
print("First 5 rows:")
print(df.head())
print("\nDataFrame info:")
df.info()
n_q = int((df["clearance"] == "?").sum())
print(f"\nMissing ('?') values in 'clearance': {n_q}")
df["clearance"] = df["clearance"].replace("?", "u")
assert (df["clearance"] == "?").sum() == 0
n_dups = int(df.duplicated().sum())
print(f"Number of duplicate rows: {n_dups}")
if n_dups:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
print(f"Cleaned dataset shape: {df.shape}")
print(df["priority"].value_counts())


### Alternate — `missing` token vs mode impute

The brief asks for `"u"`. Official RMS manuals often already use `u` for *unfounded*. Document the collision. Mode of observed clearance is usually `o` (open).


In [ ]:
raw = pd.read_csv("data/incidents.csv")
obs_mode = raw.loc[raw["clearance"] != "?", "clearance"].mode().iloc[0]
print("mode of raw clearance (excluding ?):", obs_mode)
df_missing = raw.copy()
df_missing["clearance"] = df_missing["clearance"].replace("?", "missing")
df_mode = raw.copy()
df_mode["clearance"] = df_mode["clearance"].replace("?", obs_mode)
print("levels missing-token:", sorted(df_missing["clearance"].unique()))
print("levels mode-imputed:", sorted(df_mode["clearance"].unique()))
print("We keep df with '?' → 'u' for the rest of the notebook.")


## 2. Exploratory data analysis

1. Priority balance.
2. Weapon vs priority (key feature).
3. District vs priority (overlap).
4. Factorize every column.
5. Top-12 |corr| heatmap — **12 columns only**.

Reference: `crimrisk_class_balance.png`, `crimrisk_weapon.png`, `crimrisk_district.png`, `crimrisk_heatmap.png`.

Factorize |corr| may rank `injury` above `weapon`. Gini / MI later restore weapon as the driver.


In [ ]:
fig, ax = plt.subplots()
ax = sns.countplot(data=df, x="priority", hue="priority",
                   palette={"r": "#6aa84f", "p": "#cc4125"}, legend=False, ax=ax)
ax.set_title("Priority balance: routine vs priority")
ax.set_xlabel("priority (r = routine, p = priority)")
total = len(df)
for container in ax.containers:
    labels = [f"{int(v.get_height())}\n({v.get_height()/total:.1%})" for v in container]
    ax.bar_label(container, labels=labels, padding=3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="weapon", hue="priority",
              palette={"r": "#6aa84f", "p": "#cc4125"}, ax=ax)
ax.set_title("Case priority by weapon (key feature)")
ax.set_xlabel("weapon  n=none k=knife f=firearm o=other u=unknown")
plt.tight_layout(); plt.show()

order = df["district"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="district", hue="priority", order=order,
              palette={"r": "#6aa84f", "p": "#cc4125"}, ax=ax)
ax.set_title("Case priority by district")
plt.tight_layout(); plt.show()

df_encoded = df.copy()
for col in df_encoded.columns:
    df_encoded[col], _ = pd.factorize(df_encoded[col])
print(df_encoded.head())

correlations = df_encoded.corr()["priority"].abs().drop("priority")
top12 = correlations.sort_values(ascending=False).head(12)
print("\nTop 12 features correlated with priority:")
print(top12)

top_features = list(top12.index)
plt.figure(figsize=(12, 9))
sns.heatmap(df_encoded[top_features].corr(), annot=True, fmt=".2f", cmap="Purples",
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation heatmap of top 12 features")
plt.tight_layout(); plt.show()


## 3. Preprocessing

Drop `file-flag`. LabelEncode every remaining column. 80/20, `random_state=42`. Print shapes.

Expected: `X_train (5439, 20)`, `X_test (1360, 20)`. Map **p→0, r→1**.


In [ ]:
df_model = df.drop(columns=["file-flag"])
print(f"Dropped 'file-flag'. Remaining columns: {len(df_model.columns)}")

label_encoders = {}
df_le = df_model.copy()
for col in df_le.columns:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col])
    label_encoders[col] = le
print("Encoded target classes:", dict(zip(
    label_encoders["priority"].classes_,
    label_encoders["priority"].transform(label_encoders["priority"].classes_),
)))

X = df_le.drop(columns=["priority"])
y = df_le["priority"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("test routine rate (class 1):", float(y_test.mean()))


### Alternate — one-hot for logistic regression


In [ ]:
X_oh = pd.get_dummies(df.drop(columns=["file-flag", "priority"]), drop_first=False)
y_oh = (df["priority"] == "p").astype(int)
print("one-hot width:", X_oh.shape[1])
Xoh_train, Xoh_test, yoh_train, yoh_test = train_test_split(
    X_oh, y_oh, test_size=0.2, random_state=42,
)


## 4. Random Forest

`clf = RandomForestClassifier(random_state=42)`. Fit. Predict `y_pred`.


In [ ]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("predicted routine (r=1) count:", int(y_pred.sum()))


## 5. Model evaluation

Accuracy, confusion matrix, classification report, heatmap, top-5 Gini bar.

On this seed RF ≈ **0.773**. Costly cell = actual priority (0), predicted routine (1) ≈ **164** missed priority cases.

`target_names` must follow numeric order: `['priority (p=0)', 'routine (r=1)']`.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy score:", accuracy)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print(classification_report(
    y_test, y_pred, target_names=["priority (p=0)", "routine (r=1)"],
))
print("missed priority (costly cell):", int(((y_test == 0) & (y_pred == 1)).sum()))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", square=True,
            xticklabels=["priority", "routine"],
            yticklabels=["priority", "routine"],
            cbar_kws={"shrink": 0.8})
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix — Random Forest")
plt.tight_layout(); plt.show()

importances = pd.Series(clf.feature_importances_, index=X_train.columns)
top5 = importances.sort_values(ascending=False).head(5)
print(importances.sort_values(ascending=False).head(8))
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=top5.values, y=top5.index, orient="h", color="#6d4aff")
ax.set_title("Top 5 feature importances — Random Forest")
ax.set_xlabel("Gini importance")
for i, v in enumerate(top5.values):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()


## 6. Alternate code

Decision tree, one-hot logistic regression, weapon-majority rule, mutual information.


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("DT acc", accuracy_score(y_test, dt.predict(X_test)))

lr = LogisticRegression(max_iter=2000, solver="liblinear")
lr.fit(Xoh_train, yoh_train)
print("one-hot LogReg acc", accuracy_score(yoh_test, lr.predict(Xoh_test)),
      "width", X_oh.shape[1])

maj = df.groupby("weapon")["priority"].agg(lambda s: s.value_counts().idxmax())
w_acc = float((df["weapon"].map(maj) == df["priority"]).mean())
print("weapon majority-rule acc (full table)", w_acc)
print(pd.crosstab(df["weapon"], df["priority"]))

mi = pd.Series(mutual_info_classif(X, y, random_state=42), index=X.columns).sort_values(ascending=False)
print("\nMutual information with priority:")
print(mi.head(8))


## 7. More practice

**A.** Night shift only (`time-band == n`). Does accuracy hold?

**B.** Cost matrix. A missed priority case is 8× worse than a false alarm. Sweep `predict_proba` for class 0 (priority). How many extra alarms buy fewer misses?

**C.** 2-feature card: `weapon` + `injury` only.


In [ ]:
night_code = label_encoders["time-band"].transform(["n"])[0]
night = X_test["time-band"] == night_code
print("night test rows:", int(night.sum()),
      "acc", accuracy_score(y_test[night], y_pred[night]) if night.any() else None)

# class 0 = priority, so use proba[:, 0] OR 1 - proba[:, 1]
proba_priority = clf.predict_proba(X_test)[:, 0]
print("\nmin P(priority) to flag | missed | false_alarm | acc | priority recall")
for t in [0.50, 0.40, 0.30, 0.20]:
    pred_t = np.where(proba_priority >= t, 0, 1)
    missed = int(((y_test == 0) & (pred_t == 1)).sum())
    alarm = int(((y_test == 1) & (pred_t == 0)).sum())
    acc_t = accuracy_score(y_test, pred_t)
    rec_t = recall_score(y_test, pred_t, pos_label=0)
    print(f"  {t:4.2f}                 | {missed:6d} | {alarm:11d} | {acc_t:.3f} | {rec_t:.3f}")

cols2 = ["weapon", "injury"]
clf2 = RandomForestClassifier(random_state=42)
clf2.fit(X_train[cols2], y_train)
yp2 = clf2.predict(X_test[cols2])
print("\n2-feature acc", accuracy_score(y_test, yp2),
      "missed priority", int(((y_test == 0) & (yp2 == 1)).sum()))


## 8. Simulation

| Knob | Typical movement on this table |
|------|--------------------------------|
| `max_depth` 1 → 8 | ≈ 0.74 → 0.77 |
| drop weapon | ≈ 0.62 |
| drop weapon+injury | ≈ 0.54 |
| only weapon | ≈ 0.71 |
| flip 20% / 35% labels | ≈ 0.72 / 0.63 |
| n = 50 / 800 | ≈ 0.64 / 0.75 |

Reference: `crimrisk_simulation.png`.


In [ ]:
rng = np.random.default_rng(42)
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]
fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.8))

dacc = []
for d in DEPTHS:
    m = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    dacc.append(accuracy_score(y_test, m.predict(X_test)))
labs = ["None" if d is None else str(d) for d in DEPTHS]
axes[0, 0].plot(range(len(DEPTHS)), dacc, marker="o", color="#cc4125")
axes[0, 0].set_xticks(range(len(DEPTHS))); axes[0, 0].set_xticklabels(labs)
axes[0, 0].set_title("Accuracy vs max_depth (50 trees)")
print("depth", list(zip(labs, [round(a, 4) for a in dacc])))

drop_plan = {
    "all 20": [],
    "no weapon": ["weapon"],
    "no weapon+injury": ["weapon", "injury"],
    "no weapon+offense": ["weapon", "offense"],
    "only weapon": None,
}
names, accs = [], []
for name, cols in drop_plan.items():
    if name == "only weapon":
        Xt, Xe = X_train[["weapon"]], X_test[["weapon"]]
    else:
        Xt, Xe = X_train.drop(columns=cols), X_test.drop(columns=cols)
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(Xt, y_train)
    names.append(name); accs.append(accuracy_score(y_test, m.predict(Xe)))
axes[0, 1].barh(names, accs, color="#6d4aff")
axes[0, 1].set_xlim(0.50, 1.0)
axes[0, 1].set_title("Accuracy after dropping key features")
print("drop", list(zip(names, [round(a, 4) for a in accs])))

nacc, nrec = [], []
ytr_np = y_train.to_numpy()
for f in FLIP_RATES:
    y_noisy = ytr_np.copy()
    k = int(f * len(y_noisy))
    if k:
        idx = rng.choice(len(y_noisy), size=k, replace=False)
        y_noisy[idx] = 1 - y_noisy[idx]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train, y_noisy)
    yp = m.predict(X_test)
    nacc.append(accuracy_score(y_test, yp))
    nrec.append(recall_score(y_test, yp, pos_label=0))
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nacc, marker="o", label="accuracy", color="#6aa84f")
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nrec, marker="s", label="priority recall", color="#cc4125")
axes[1, 0].set_title("Train label-flip vs test metrics")
axes[1, 0].legend()
print("noise acc", list(zip(FLIP_RATES, [round(a, 4) for a in nacc])))

sacc = []
perm = rng.permutation(len(X_train))
for n in TRAIN_NS:
    take = perm[:n]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train.iloc[take], y_train.iloc[take])
    sacc.append(accuracy_score(y_test, m.predict(X_test)))
axes[1, 1].plot(TRAIN_NS, sacc, marker="o", color="#e69138")
axes[1, 1].set_title("Accuracy vs random training n")
print("n-sub", list(zip(TRAIN_NS, [round(a, 4) for a in sacc])))
plt.suptitle("CrimRisk simulation knobs")
plt.tight_layout(); plt.show()


## 9. Implementation checklist (step by step)

Full printable version: **`CrimRisk_Implementation_Checklist.docx`**. Tick these before a live RMS job.

**Phase A — Scope**
1. Write the unit of analysis in one sentence: *incident row*, not person.
2. Name the costly error: missed priority case.
3. List fields that are banned (race, ethnicity, religion, raw address → tract).
4. Name the human who will override the flag.

**Phase B — Data**
5. Inventory source tables and join keys.
6. Count `"?"` / blank / `NULL` per column; choose *unknown token* vs drop vs mode — write the choice down.
7. Deduplicate on the incident number, not on the feature vector alone.
8. Drop zero-variance columns (`file-flag` here).
9. Freeze a time-based split for production; this lab uses a random 80/20 only because the extract has no date.

**Phase C — Model**
10. Fit a one-feature rule (weapon) as the baseline you must beat.
11. Fit RF + one alternate (DT or one-hot LR).
12. Report the costly cell, not only accuracy.
13. Sweep the probability threshold against *reviewer capacity* (how many extra cases per week).

**Phase D — Review**
14. Analyst reads the memo for the commander and the public versions.
15. Legal / policy sign-off: not bail, not sentencing, not charging.
16. Shadow mode for N weeks: model scores, humans decide, compare costly cells.
17. Log every override. Retrain only on audited labels.
18. Kill switch: if missingness or offense mix drifts, stop writing flags.

Run the cheap audit below on this teaching table.


In [ ]:
audit = {
    "unit_is_incident": True,
    "banned_fields_present": False,
    "unknown_policy": "clearance ? → u (documented collision with unfounded)",
    "dups_dropped": 11,
    "constant_dropped": "file-flag",
    "baseline_acc": round(w_acc, 3),
    "model_acc": round(float(accuracy), 3),
    "missed_priority": int(((y_test == 0) & (y_pred == 1)).sum()),
    "human_override": "desk sergeant / crime analyst (named in runbook)",
    "production_split": "NOT this random 80/20 — use incident date",
}
print(pd.Series(audit))
print("\nChecklist gate:", "PASS teaching lab" if audit["missed_priority"] is not None else "FAIL")


## 10. Audience notes

| Audience | Show |
|----------|------|
| Expert (crime analyst / researcher) | MI vs Gini vs factorize, p=0 map, why 0.77 is not a risk score |
| Technician (RMS / CAD implementer) | 2-feature card, threshold = desk capacity, do not write to the charging file |
| Executive (commander / prosecutor admin) | 50/50 balance, 0.77 acc, 164 missed priority on hold-out, human review required |
| Nonspecialist (public / reporter) | “weapon is the loud clue *in this practice book*, not a verdict about a neighborhood” |

Full prose: `CrimRisk_Project_Memo.docx`.



## 11. Good fit vs limitations

**Good fit:** all-categorical incident codes, nearly balanced priority flag, costly FN, teaching clean vs impute.

**Anti-applications:** bail, pretrial release, sentencing, charging, stop-and-frisk targeting, gang-database scoring, person-level recidivism. Do not add demographic proxies and rerun.

The pattern (categorical RF + costly FN + checklist) transfers to *case triage* only — always with an analyst in the loop.

